# GitHub deploy role

Lets GitHub Actions deploy to AWS **without any stored AWS keys**.
Creates two things:

| Item | What it is |
|---|---|
| **OIDC identity provider** for GitHub | Tells AWS: "tokens issued by `token.actions.githubusercontent.com` are trusted identity proofs" |
| **Role `pf-github-deploy-role`** | The only identity GitHub can become. Its trust policy accepts a token only from **this repository's `main` branch**. Its permissions cover only the Glue scripts folder, the Glue job and the ingest Lambda. |

Run the cells top to bottom. Each cell is safe to run twice, and prints the AWS CLI command it runs (`$ aws ...`).

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# aws() runs the AWS CLI and prints the command and its output (see infra/aws_helpers.py)
from aws_helpers import SETTINGS, LAKE, REGION, account_id, aws, aws_json, render_policy, whoami

ROLE = "pf-github-deploy-role"
REPOSITORY = SETTINGS["github_repository"]
PROVIDER_ARN = f"arn:aws:iam::{account_id()}:oidc-provider/token.actions.githubusercontent.com"

print("identity  :", whoami())
print("repository:", REPOSITORY)

identity  : user/Ahmed
repository: Abdul-wahab456/aws-patient-flow-pipeline


## 2. OIDC identity provider
One per AWS account. GitHub signs a short-lived token for every workflow run; this provider is what lets AWS
check that signature.

In [2]:
exists = aws("iam", "get-open-id-connect-provider", "--open-id-connect-provider-arn", PROVIDER_ARN,
             check=False, show=False).returncode == 0
if exists:
    print("provider already exists - skipping")
else:
    aws("iam", "create-open-id-connect-provider", "--url", "https://token.actions.githubusercontent.com",
        "--client-id-list", "sts.amazonaws.com", "--query", "OpenIDConnectProviderArn", "--output", "text")

$ aws iam get-open-id-connect-provider --open-id-connect-provider-arn arn:aws:iam::<ACCOUNT_ID>:oidc-provider/token.actions.githubusercontent.com


provider already exists - skipping


## 3. The deploy role
- **Trust policy** (`policy_github_deploy.template.json`): only a token whose subject is
  `repo:<owner>/<repository>:ref:refs/heads/main` can assume the role. Another repository, a fork,
  or another branch is refused.
- **Permissions** (`permissions_github_deploy.template.json`): upload to `glue_scripts/`, manage the one Glue job,
  update the one Lambda, and hand the Glue and Lambda roles to those services. Nothing else.

In [3]:
description = "GitHub Actions: deploy the Glue job and the ingest Lambda"
trust = render_policy("policy_github_deploy.template.json")
exists = aws("iam", "get-role", "--role-name", ROLE, check=False, show=False).returncode == 0

if exists:
    print(f"role {ROLE} already exists - updating its trust policy\n")
    aws("iam", "update-assume-role-policy", "--role-name", ROLE, "--policy-document", trust)
else:
    aws("iam", "create-role", "--role-name", ROLE, "--description", description,
        "--assume-role-policy-document", trust, "--query", "Role.Arn", "--output", "text")

aws("iam", "tag-role", "--role-name", ROLE, "--tags", "Key=project,Value=patient-flow")
aws("iam", "update-role", "--role-name", ROLE, "--description", description)
aws("iam", "put-role-policy", "--role-name", ROLE, "--policy-name", f"{ROLE}-permissions",
    "--policy-document", render_policy("permissions_github_deploy.template.json"))

$ aws iam get-role --role-name pf-github-deploy-role


role pf-github-deploy-role already exists - updating its trust policy

$ aws iam update-assume-role-policy --role-name pf-github-deploy-role --policy-document {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Federated": "arn:aws:iam::<ACCOUNT_ID>:oidc-provider/token.actions.githubusercontent.com"}, "Action": "sts:AssumeRoleWithWebIdentity", "Condition": {"StringEquals": {"token.actions.githubusercontent.com:aud": "sts.amazonaws.com", "token.actions.githubusercontent.com:sub": "repo:Abdul-wahab456/aws-patient-flow-pipeline:ref:refs/heads/main"}}}]}


$ aws iam tag-role --role-name pf-github-deploy-role --tags Key=project,Value=patient-flow


$ aws iam update-role --role-name pf-github-deploy-role --description GitHub Actions: deploy the Glue job and the ingest Lambda


$ aws iam put-role-policy --role-name pf-github-deploy-role --policy-name pf-github-deploy-role-permissions --policy-document {"Version": "2012-10-17", "Statement": [{"Sid": "ListScriptsFolder", "Effect": "Allow", "Action": "s3:ListBucket", "Resource": "arn:aws:s3:::patientflow-datalake-zhmkbf", "Condition": {"StringLike": {"s3:prefix": ["glue_scripts/*"]}}}, {"Sid": "UploadGlueScripts", "Effect": "Allow", "Action": "s3:PutObject", "Resource": "arn:aws:s3:::patientflow-datalake-zhmkbf/glue_scripts/*"}, {"Sid": "ManageTheGlueJob", "Effect": "Allow", "Action": ["glue:GetJob", "glue:CreateJob", "glue:UpdateJob", "glue:TagResource"], "Resource": "arn:aws:glue:us-east-1:<ACCOUNT_ID>:job/pf-bronze-to-silver"}, {"Sid": "UpdateTheIngestLambda", "Effect": "Allow", "Action": ["lambda:GetFunction", "lambda:GetFunctionConfiguration", "lambda:UpdateFunctionCode", "lambda:UpdateFunctionConfiguration"], "Resource": "arn:aws:lambda:us-east-1:<ACCOUNT_ID>:function:pf-ingest"}, {"Sid": "HandTheGlueRoleT

<aws exit code 0>

## 4. Verify the trust
Read the trust policy back from AWS: it must name the provider and exactly this repository's main branch.

In [4]:
role = aws_json("iam", "get-role", "--role-name", ROLE)["Role"]
statement = role["AssumeRolePolicyDocument"]["Statement"][0]
condition = statement["Condition"]["StringEquals"]

assert statement["Principal"]["Federated"] == PROVIDER_ARN
assert statement["Action"] == "sts:AssumeRoleWithWebIdentity"
assert condition["token.actions.githubusercontent.com:aud"] == "sts.amazonaws.com"
assert condition["token.actions.githubusercontent.com:sub"] == f"repo:{REPOSITORY}:ref:refs/heads/main"
print("OK  only tokens of", condition["token.actions.githubusercontent.com:sub"], "can use this role")

OK  only tokens of repo:Abdul-wahab456/aws-patient-flow-pipeline:ref:refs/heads/main can use this role


## 5. Verify the permissions
The policy simulator answers "would this role be allowed to do X?" without doing it.
We check what the deployment needs, and what it must **not** be able to do.

In [5]:
ROLE_ARN = role["Arn"]
S3 = f"arn:aws:s3:::{LAKE}"
ACCOUNT = account_id()
JOB_ARN = f"arn:aws:glue:{REGION}:{ACCOUNT}:job/{SETTINGS['glue_job_name']}"
LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT}:function:{SETTINGS['lambda_name']}"
def role_arn(name):
    return f"arn:aws:iam::{ACCOUNT}:role/{name}"



def pass_to(service):
    return [f"ContextKeyName=iam:PassedToService,ContextKeyValues={service},ContextKeyType=string"]


# (action, resource, extra simulator arguments, expected decision)
CHECKS = [
    ("s3:PutObject", f"{S3}/glue_scripts/abc123/bronze_to_silver.py", [], "allowed"),
    ("glue:UpdateJob", JOB_ARN, [], "allowed"),
    ("lambda:UpdateFunctionCode", LAMBDA_ARN, [], "allowed"),
    ("iam:PassRole", role_arn("pf-glue-role"), pass_to("glue.amazonaws.com"), "allowed"),
    ("iam:PassRole", role_arn("pf-lambda-role"), pass_to("lambda.amazonaws.com"), "allowed"),
    # what a deployment must NOT be able to do
    ("s3:PutObject", f"{S3}/bronze/patient_records/x.json", [], "implicitDeny"),
    ("s3:PutObject", f"{S3}/silver/patient/x.parquet", [], "implicitDeny"),
    ("s3:GetObject", f"{S3}/landing/batch_01/x.json", [], "implicitDeny"),
    ("s3:DeleteObject", f"{S3}/glue_scripts/abc123/bronze_to_silver.py", [], "implicitDeny"),
    ("glue:UpdateJob", f"arn:aws:glue:{REGION}:{ACCOUNT}:job/some-other-job", [], "implicitDeny"),
    ("lambda:UpdateFunctionCode", f"arn:aws:lambda:{REGION}:{ACCOUNT}:function:some-other-function", [], "implicitDeny"),
    ("iam:CreateRole", role_arn("anything"), [], "implicitDeny"),
    ("iam:PassRole", role_arn("pf-step-functions-role"), pass_to("states.amazonaws.com"), "implicitDeny"),
    ("iam:PassRole", role_arn("pf-glue-role"), pass_to("lambda.amazonaws.com"), "implicitDeny"),
]

failures = []
for action, resource, extra, expected in CHECKS:
    args = ["iam", "simulate-principal-policy", "--policy-source-arn", ROLE_ARN,
            "--action-names", action, "--resource-arns", resource]
    if extra:
        args += ["--context-entries", *extra]
    decision = aws_json(*args)["EvaluationResults"][0]["EvalDecision"]
    shown = resource.replace(LAKE, "<lake>").replace(ACCOUNT, "<ACCOUNT_ID>")
    print(f"{'OK  ' if decision == expected else 'FAIL'} {action:<26} {shown:<70} {decision}")
    if decision != expected:
        failures.append(action)

assert not failures, f"{len(failures)} permission checks failed"
print(f"\nall {len(CHECKS)} permission checks passed")

OK   s3:PutObject               arn:aws:s3:::<lake>/glue_scripts/abc123/bronze_to_silver.py            allowed


OK   glue:UpdateJob             arn:aws:glue:us-east-1:<ACCOUNT_ID>:job/pf-bronze-to-silver            allowed


OK   lambda:UpdateFunctionCode  arn:aws:lambda:us-east-1:<ACCOUNT_ID>:function:pf-ingest               allowed


OK   iam:PassRole               arn:aws:iam::<ACCOUNT_ID>:role/pf-glue-role                            allowed


OK   iam:PassRole               arn:aws:iam::<ACCOUNT_ID>:role/pf-lambda-role                          allowed


...

OK   glue:UpdateJob             arn:aws:glue:us-east-1:<ACCOUNT_ID>:job/some-other-job                 implicitDeny


OK   lambda:UpdateFunctionCode  arn:aws:lambda:us-east-1:<ACCOUNT_ID>:function:some-other-function     implicitDeny


OK   iam:CreateRole             arn:aws:iam::<ACCOUNT_ID>:role/anything                                implicitDeny


OK   iam:PassRole               arn:aws:iam::<ACCOUNT_ID>:role/pf-step-functions-role                  implicitDeny


OK   iam:PassRole               arn:aws:iam::<ACCOUNT_ID>:role/pf-glue-role                            implicitDeny

all 14 permission checks passed


## What to set in GitHub
In the repository: **Settings > Secrets and variables > Actions > Variables > New repository variable**.
These are names, not secrets. The role ARN contains the account id, so it is not printed here; get it in a terminal with

`aws iam get-role --role-name pf-github-deploy-role --query Role.Arn --output text`

| Variable | Value |
|---|---|
| `AWS_DEPLOY_ROLE_ARN` | the ARN printed by the command above |
| `AWS_REGION` | `us-east-1` |
| `LAKE_BUCKET` | the data lake bucket name from `config/settings.json` |
| `SILVER_DATABASE` | `pf_silver` |
| `NUMBER_OF_BATCHES` | `4` |

**Next:** push the repository; the `Deploy` workflow then deploys on every push to `main`.